# FBM + EDS WL Residual Map Fusion Briefing

이 노트북은 팀원 브리핑용 설명자료입니다. 실제 데이터 분석 결과를 담는 노트북이 아니라, 실제 FBM tensor dataset과 EDS tabular dataset을 현재 repo 구조에 연결할 때의 **모델 아키텍처, 데이터 계약, 평가 컨셉, leakage guardrail**을 설명합니다.

Validation status는 마지막 `## Checks` 섹션에서 확인합니다.

## Goal

우리가 만들려는 구조는 다음입니다.

- FBM image tensor는 기존 image branch로 학습합니다.
- EDS scalar tabular feature는 CatBoost one-vs-rest OOF logits로 바꿔 fusion input으로 사용합니다.
- EDS test item 중 wordline 위치가 매핑되는 항목은 train real population 기준 high-side residual map으로 바꿔 WL branch에 사용합니다.
- Synthetic composite에는 raw tabular row를 만들지 않습니다. parent residual map의 max composition과 union mask/source count만 낮은 weight로 사용합니다.
- 공식 평가는 real sample, 특히 real composite 기준으로 유지합니다.

## Setup

이 노트북은 외부 데이터 파일 없이 실행됩니다. 표와 체크리스트를 생성하기 위해 `pandas`만 사용합니다.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
print("briefing notebook ready")

## Steps

### 1. 입력 데이터 계약

실제 리팩토링은 데이터 계약을 먼저 맞추는 것이 핵심입니다. 아래 표는 repo에서 기대하는 canonical input입니다.

In [ ]:
dataset_contract = pd.DataFrame([
    {
        "dataset": "FBM image tensor",
        "path": "data/raw/fbm_tensor/fbm_images.npy",
        "required_keys_or_columns": "shape [N,H,W] or [N,C,H,W]",
        "purpose": "image branch input",
    },
    {
        "dataset": "FBM manifest",
        "path": "data/raw/fbm_tensor/fbm_manifest.csv",
        "required_keys_or_columns": "row_idx, sample_id, split, eval_group, label_*",
        "purpose": "image row alignment, labels, official eval group",
    },
    {
        "dataset": "EDS tabular",
        "path": "data/raw/eds_tabular/eds_tabular.parquet or .csv",
        "required_keys_or_columns": "sample_id, split, eval_group, label_*, EDS feature columns",
        "purpose": "CatBoost input and WL measurement source",
    },
    {
        "dataset": "EDS item to WL map",
        "path": "data/metadata/eds_test_item_wordline_map.csv",
        "required_keys_or_columns": "feature_name, test_method, test_item, wordline or WL range, value_direction",
        "purpose": "wide EDS columns -> long WL measurements",
    },
])
dataset_contract

### 2. 모델 아키텍처

```text
FBM image tensor
  -> FBM encoder
  -> fbm_logits

EDS raw scalar features
  -> CatBoost one-vs-rest models
  -> OOF / fold-ensemble catboost_logits

EDS wordline-resolved measurements
  -> train-real median/IQR baseline
  -> high-side residual map [C, WL bins, test methods]
  -> WL map encoder
  -> wl_logits

fbm_logits + has_wl_map * gate_wl * wl_logits + has_catboost_logits * gate_cat * catboost_logits
  -> class-wise gated residual fusion
  -> fusion_logits
```

Gate는 class-wise입니다. 즉 WL branch가 특정 class에는 강하고 다른 class에는 약할 수 있는 구조를 허용합니다.

### Architecture Diagram

아래 그림은 세 입력 branch와 loss 연결을 한 장으로 요약합니다. CatBoost는 neural loss를 만들지 않고, real EDS tabular로 offline 학습된 logits만 fusion에 제공합니다.

![FBM + WL Residual Map + CatBoost Fusion Architecture](../docs/assets/model_architecture.svg)

In [ ]:
architecture_stages = pd.DataFrame([
    {"stage": "FBM branch", "input": "fbm_images.npy", "output": "fbm_logits", "training_note": "real + synthetic image samples can train image head"},
    {"stage": "WL residual branch", "input": "wl_measurements.parquet", "output": "wl_logits", "training_note": "baseline fit must use train real only"},
    {"stage": "CatBoost branch", "input": "EDS scalar features", "output": "catboost_logits", "training_note": "train predictions must be OOF; synthetic excluded"},
    {"stage": "Fusion", "input": "fbm_logits, wl_logits, catboost_logits + masks", "output": "fusion_logits", "training_note": "missing modalities use zero logits plus mask"},
])
architecture_stages

### 3. EDS Test Item to Wordline Mapping

Mapping table은 EDS tabular feature column을 WL 위치로 연결합니다. 원칙은 **one row per EDS feature column**입니다.

팀원이 직접 작성할 핵심 컬럼은 단순하게 유지합니다.

```text
feature_name, eds_step, eds_item, wordline_position
```

모델 routing flag는 validator가 자동으로 만듭니다.

- `wordline_position`이 있으면 `include_in_wl_map=1`
- `wordline_position`이 비어 있으면 `include_in_wl_map=0`
- numeric EDS feature는 기본 `include_in_catboost=1`
- CatBoost에서 제외할 feature만 mapping table에 `include_in_catboost=0`을 명시합니다.
- `low_bad` feature는 WL residual 변환 전에 value sign을 뒤집어 high-side residual 정의에 맞춥니다.

In [ ]:
mapping_schema = pd.DataFrame([
    {"column": "feature_name", "required": True, "example": "EDS_RD_WL000", "meaning": "exact EDS tabular column name"},
    {"column": "eds_step", "required": True, "example": "READ", "meaning": "test-method axis in WL map"},
    {"column": "eds_item", "required": True, "example": "RD_LEAK", "meaning": "engineer-facing EDS item name"},
    {"column": "wordline_position", "required": True, "example": "0 or WL000", "meaning": "single WL index; blank for global scalar features"},
    {"column": "value_direction", "required": False, "example": "high_bad", "meaning": "high_bad by default; low_bad values are sign-flipped for residual maps"},
    {"column": "include_in_catboost", "required": False, "example": "1", "meaning": "optional override; defaults to 1 for numeric EDS features"},
])
mapping_schema

### 4. Residual Map Concept

For each train-real baseline group `(test_method, wl_bin)`:

```text
R = max(0, (value - median_train[test_method, wl_bin]) / (IQR_train[test_method, wl_bin] + eps))
```

Interpretation:

- `R=0`: train population median보다 높게 튀지 않음
- `R>0`: 같은 test method / WL bin에서 high-side abnormality가 있음
- value channels: mean/max/std residual
- mask channels: observed mask, count ratio, source count

Validation/test는 baseline fit에 들어가면 안 됩니다.

### 5. Evaluation Concept

공식 metric은 synthetic을 섞지 않습니다. Synthetic sample은 학습 보조와 diagnostic에만 사용합니다.

```text
official:
  real_single
  real_composite
  real_all

auxiliary only:
  synthetic_composite
```

Fusion gain은 real-only KPI에서 계산합니다.

In [ ]:
evaluation_groups = pd.DataFrame([
    {"group": "real_single", "official_metric": True, "why": "single-defect real behavior"},
    {"group": "real_composite", "official_metric": True, "why": "target scarce composite behavior"},
    {"group": "real_all", "official_metric": True, "why": "combined real summary"},
    {"group": "synthetic_composite", "official_metric": False, "why": "auxiliary only; not real production evidence"},
])
evaluation_groups

### 6. Leakage Guardrails

브리핑에서 가장 강조해야 할 risk controls입니다.

- WL median/IQR baseline은 train real sample만 사용합니다.
- CatBoost train logits는 in-fold prediction이 아니라 OOF prediction만 사용합니다.
- Synthetic sample은 CatBoost training에서 제외합니다.
- Synthetic sample은 official validation/test metric에 포함하지 않습니다.
- Pseudo-labeling은 기본 off입니다.
- Pseudo-labeling을 켤 경우 global top-K 금지, pairwise top-K만 사용합니다.

In [ ]:
handoff_checklist = pd.DataFrame([
    {"check": "FBM and EDS sample_id join", "owner": "data loader", "pass_condition": "no duplicate sample_id; missing modality masks explicit"},
    {"check": "EDS mapping coverage", "owner": "mapping validator", "pass_condition": "wordline_position derives WL-map routing correctly"},
    {"check": "WL baseline scope", "owner": "WL cache builder", "pass_condition": "fit_sample_ids subset of train_real_sample_ids"},
    {"check": "CatBoost OOF", "owner": "CatBoost script", "pass_condition": "train_prediction_mode == OOF"},
    {"check": "Official metric scope", "owner": "evaluator", "pass_condition": "synthetic_as_auxiliary_only"},
])
handoff_checklist

## Checks

이 노트북 자체의 체크:

- 외부 데이터 없이 실행 가능해야 합니다.
- `pandas` 표가 출력되어야 합니다.
- 실제 성능 수치나 데이터 기반 결론을 주장하지 않습니다.
- 실제 리팩토링 source of truth는 `docs/real_dataset_roo_refactor_guide.md`입니다.

In [ ]:
assert dataset_contract.shape[0] == 4
assert mapping_schema["column"].is_unique
assert evaluation_groups["official_metric"].tolist() == [True, True, True, False]
assert "WL baseline scope" in set(handoff_checklist["check"])
print("Notebook checks passed")

## Next Steps

1. `data/metadata/eds_test_item_wordline_map.csv`를 먼저 작성하고 엔지니어 검수를 받습니다.
2. `validate-eds-map` CLI로 mapping coverage와 WL/CatBoost routing count를 확인합니다.
3. `build-wl-measurements` CLI로 `data/interim/wl_measurements.csv`를 생성합니다.
4. `fbm_multimodal.fusion.real_data` loader로 FBM tensor와 EDS tabular join smoke test를 수행합니다.
5. WL residual tensor cache와 CatBoost OOF logits를 생성합니다.
6. real-only official metric과 leakage report를 팀 리뷰에 공유합니다.